In [1]:
import json
import re
import pandas as pd
from pathlib import Path
from io import StringIO
from bs4 import BeautifulSoup
from dateutil.parser import parse

# 1. Clean / classify columns
# 2. Find <statements> column
# 3. Move repeated-span noise rows below
# 4. Detect audit / date / QT / FY
# 5. Move those detected header rows below
# 6. NOW traverse the cleaned main data
#    ├── <statements> + right side empty → NO DATA
#    ├── <statements> + right side has data → DATA
#    └── detect a), 1., i), (a), etc.
# 7. Add/update flags

AUDIT_RE = re.compile(r"\b(un\-?audited|audited)\b", re.I)

DATE_RE = re.compile(
    r"""
    (?<!\d)
    (
        # 30-06-2026 / 30.06.2026 / 30/06/2026
        \d{1,2}[./-]\d{1,2}[./-]\d{2,4}

        |

        # 30-Jun-26 / 30-June-2026
        \d{1,2}[./-]
        (?:jan|january|feb|february|mar|march|apr|april|
        may|jun|june|jul|july|aug|august|sep|sept|september|
        oct|october|nov|november|dec|december)
        [./-]\d{2,4}

        |

        # 30 Jun 2026
        \d{1,2}\s+
        (?:jan|january|feb|february|mar|march|apr|april|
        may|jun|june|jul|july|aug|august|sep|sept|september|
        oct|october|nov|november|dec|december)
        \s+\d{2,4}

        |

        # Jun 2026
        (?:jan|january|feb|february|mar|march|apr|april|
        may|jun|june|jul|july|aug|august|sep|sept|september|
        oct|october|nov|november|dec|december)
        \s+\d{2,4}
    )
    (?!\d)
    """,
    re.I | re.X,
)

NUM_LINE_RE = re.compile(
    r"""
    ^\s*
    [(\[{]?\s*              # optional opening bracket
    [₹$€£]?\s*              # optional currency
    [-+]?\s*                # optional sign
    \d[\d,]*                # integer: 1, 123, 1,234
    (?:\.\d+)?              # optional decimal
    %?                      # optional percentage
    \s*[)\]}]?              # optional closing bracket
    \s*$
    """,
    re.VERBOSE,
)

# NUM_LINE_RE = re.compile(
#     r"""
#     ^\s*
#     (?!\d{1,3}\s*$)          # exclude standalone integers: 1, 2, 3, 1., 2.
#     [\(\[\{]?
#     [₹$€£]?
#     [-+]?
#     \d[\d,]*
#     (?:\.\d+)?
#     %?
#     [\)\]\}]?
#     \s*$
#     """,
#     re.VERBOSE,
# )

QT_RE = re.compile(r"\b(?:quarter|quarter\s+ended|3[-\s]?months?(?:\s+ended)?|three[-\s]?months?(?:\s+ended)?)\b",re.I)
FY_RE = re.compile(r"\b(?:year(?:\s+ended)?|12[-\s]?months?(?:\s+ended)?|twelve[-\s]?months?(?:\s+ended)?)\b",re.I)
PARTICULARS_RE = re.compile(r"[A-Za-z\s]{4,}")
PARTICULAR_SUFFIX_RE = re.compile(r"^\s*(?:[A-Za-z]\)|\d+\.|[ivxlcdm]+\)|\([A-Za-z]\))", re.I)

PREFIX_RE = re.compile(
    r"""
    ^\s*
    (
        \([ivxlcdm]+\)   |  # (i), (ii)
        [ivxlcdm]+\)     |  # i), ii)
        \([a-z]\)        |  # (a), (b)
        [a-z][.)]        |  # a., a), b., b)
        \d+[.)]?            # 1, 1., 1)
    )
    \s*
    """,
    re.I | re.X
)

In [3]:
def normalize_financial_dataframe(df, top_fraction=0.20):

    df = df.copy().fillna("").map(lambda x: " ".join(str(x).split()))

    metadata = {
        j: {
            "column_type": None,
            "audit": "",
            "date": "",
            "period_type": "",
            "period": "",
        }
        for j in range(df.shape[1])
    }

    start = int(len(df) * top_fraction)

    # 1. COLUMN CLASSIFICATION

    for j in range(df.shape[1]):

        values = df.iloc[start:, j].astype(str).str.strip()
        non_empty = values[values != ""]
        numeric_ratio = (
            non_empty
            .map(lambda x: bool(NUM_LINE_RE.fullmatch(x)))
            .astype(float)
            .mean()
        )
        particulars_ratio = (
            non_empty
            .map(lambda x: bool(PARTICULARS_RE.findall(x)))
            .astype(float)
            .mean()
        )

        if numeric_ratio >= 0.4:
            column_type = "<numeric>"

        elif particulars_ratio >= 0.4:
            column_type = "<statements>"

        else:
            column_type = "OTHER"

        metadata[j]["column_type"] = column_type

    particular_col = next((j for j in metadata if metadata[j]["column_type"] == "<statements>"), None)

    if particular_col is not None:
        noise_rows = []
        kept_rows = []
        for i in range(len(df)):
            row = df.iloc[i]
            values = row.iloc[particular_col:].astype(str).str.strip()
            non_empty = values[values != ""]
            if len(non_empty) > 1 and non_empty.nunique() == 1:
                new_row = [""] * df.shape[1]
                new_row[0] = non_empty.iloc[0]
                noise_rows.append(new_row)
            else:
                kept_rows.append(row.tolist())
                
        

        df = pd.DataFrame(kept_rows + noise_rows, columns=df.columns).reset_index(
            drop=True
        )
        start = int(len(df) * top_fraction)
        

        

    for j in range(df.shape[1]):
        for i in range(start):
            text = str(df.iloc[i, j]).strip()
            match = AUDIT_RE.findall(text)
            if match:
                audit = match[0].lower()
                metadata[j]["audit"] = (
                    "Unaudited" if "unaudit" in audit.replace("-", "") else "Audited"
                )
                break

    for j in range(df.shape[1]):
        for i in range(start):
            text = str(df.iloc[i, j]).strip()
            match = DATE_RE.findall(text)
            if match:
                metadata[j]["date"] = match[0]
                break

    for i in range(start):
        for j in range(df.shape[1]):

            text = str(df.iloc[i, j]).strip()
            if QT_RE.findall(text):
                metadata[j]["period_type"] = "QT"
            elif FY_RE.findall(text):
                metadata[j]["period_type"] = "FY"
                
            

    # ROW-WISE <statements> CHECK
    particular_col = next((j for j in metadata if metadata[j]["column_type"] == "<statements>"),None)
    numeric_cols = [
        j for j in metadata
        if metadata[j]["column_type"] == "<numeric>"
    ]

    df["ROW_TYPE"] = ""
    if particular_col is not None:
        for i in range(len(df)):
            particular = str(df.iloc[i, particular_col]).strip()
            if not particular:
                continue

            values = df.iloc[i, numeric_cols].astype(str).str.strip()
            if values.eq("").all():
                df.loc[i, "ROW_TYPE"] = "NO DATA"
            else:
                df.loc[i, "ROW_TYPE"] = "DATA"
                

    df["PREFIX"] = ""
    if particular_col is not None:
        for i in range(len(df)):
            particular = str(df.iloc[i, particular_col]).strip()
            match = PREFIX_RE.match(particular)

            if match:
                df.loc[i, "PREFIX"] = match.group()
                df.iloc[i, particular_col] = particular[match.end():].strip()


        cols = list(df.columns)
        cols.remove("PREFIX")
        cols.insert(particular_col, "PREFIX")
        df = df[["ROW_TYPE"] + [c for c in cols if c != "ROW_TYPE"]]

    # MOVE DETECTED HEADER ROWS TO BOTTOM
    move_indices = set()
    for i in range(start):
        for j in range(df.shape[1]):
            text = str(df.iloc[i, j]).strip()

            if (
                AUDIT_RE.findall(text)
                or DATE_RE.findall(text)
                or QT_RE.findall(text)
                or FY_RE.findall(text)
            ):
                move_indices.add(i)
                break

    moved_rows = df.iloc[sorted(move_indices)].copy()
    df = df.drop(index=move_indices)
    
    separator = pd.DataFrame(
        [["="] * df.shape[1]] * 2,
        columns=df.columns
    )
    df = pd.concat([df,separator, moved_rows], ignore_index=True)
        
    
    # metadata_df = pd.DataFrame(
    #     [
    #         [metadata[j]["column_type"] for j in range(df.shape[1])],
    #         [metadata[j]["audit"] for j in range(df.shape[1])],
    #         [metadata[j]["date"] for j in range(df.shape[1])],
    #         [metadata[j]["period_type"] for j in range(df.shape[1])],
    #     ],
    #     columns=df.columns,
    # )
    metadata_df = pd.DataFrame(
        [
            [""]*2 + [metadata[j]["column_type"] for j in metadata] ,
            [""]*2 + [metadata[j]["audit"] for j in metadata] ,
            [""]*2 + [metadata[j]["date"] for j in metadata] ,
            [""]*2 + [metadata[j]["period_type"] for j in metadata] ,
        ],
        columns=df.columns
    )
        
    

    df = pd.concat([metadata_df, df], ignore_index=True)
    return df, metadata


In [ ]:
JSON_DIR = Path(r"C:\Users\kaustubh.keny\Downloads\OFFICE_FILES\outputs")
BASE_HTML = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Extracted Tables</title>
<style>
body { font-family: Arial, sans-serif; margin: 30px; }
.source { font-size: 18px; font-weight: bold; margin: 40px 0 20px; }
.figure-title { font-size: 16px; font-weight: bold; margin: 20px 0 10px; }
table { border-collapse: collapse; margin-bottom: 40px; }
td, th { border: 1px solid #999; padding: 6px; }
</style>
</head>
<body>
<h1>Extracted Tables</h1>
"""

# After identifying the <statements> column, we will later check each row to see whether all cells to its right are empty, helping us identify possible section headers.
# Remove the original row containing the Audited/Unaudited values after detection, as these audit labels will always appear on the same row.

def clean_table(table_html):
    soup = BeautifulSoup(table_html, "html.parser")
    table = soup.find("table")

    if not table:
        return table_html

    total_cols = max(
        sum(int(c.get("colspan", 1)) for c in r.find_all(["td", "th"], recursive=False))
        for r in table.find_all("tr")
    )

    noise_rows = []

    for row in list(table.find_all("tr")):
        cells = row.find_all(["td", "th"], recursive=False)

        if not cells or all(not c.get_text(strip=True) for c in cells):
            row.decompose()
            continue

        if len(cells) == 1 and int(cells[0].get("colspan", 1)) >= total_cols:
            noise_rows.append(cells[0].get_text(" ", strip=True))
            row.decompose()
            continue

        for cell in cells:
            parts = [x.strip() for x in cell.get_text().splitlines() if x.strip()]
            cell.clear()

            for i, part in enumerate(parts):
                if i:
                    cell.append(soup.new_tag("br"))
                cell.append(part)

    result = str(table)

    if noise_rows:
        result += "<br><br><hr><br>" + "".join(
            f"<div>{text}</div>" for text in noise_rows
        )

    return result

def process_json(json_file):
    print(f"Processing: {json_file.name}")

    all_dfs = {}
    html = BASE_HTML

    html_out = JSON_DIR / f"{json_file.stem}.html"
    xls_out = JSON_DIR / f"{json_file.stem}.xlsx"

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for page_data in data["pages"]:
        pdf_name = data["batch"]
        page_num = page_data["page"]

        current_title = None
        page_has_table = False
        table_num = 0

        for block in page_data["blocks"]:
            label = str(block.get("label", "")).lower().strip()
            content = block.get("content", "")

            if label == "figure_title":
                current_title = content
                continue

            if label != "table":
                continue

            table_num += 1

            if not page_has_table:
                html += (
                    f'<div class="source">'
                    f"{pdf_name} | Page {page_num}"
                    f"</div>"
                )
                page_has_table = True

            if current_title:
                html += (
                    f'<div class="figure-title">'
                    f"{current_title}"
                    f"</div>"
                )
                current_title = None

            table = clean_table(content)
            html += f"<div>{table}</div>"

            try:
                tables = pd.read_html(StringIO(table))

                if not tables:
                    print(
                        f"  No readable table found "
                        f"on page {page_num}, table {table_num}"
                    )
                    continue

                df = tables[0]

            except (ValueError, ImportError) as exc:
                print(
                    f"  Failed to parse table "
                    f"on page {page_num}, table {table_num}: {exc}"
                )
                continue

            df = (
                df.fillna("")
                .map(lambda x: " ".join(str(x).split()))
            )


            normalized_df, metadata = normalize_financial_dataframe(df)

            sheet_name = f"page{page_num}_table_{table_num}"
            all_dfs[sheet_name] = normalized_df

    html += "</body></html>"

    html_out.write_text(html, encoding="utf-8")
    print(f"Saved HTML: {html_out}")

    if not all_dfs:
        print(f"No tables found in {json_file.name}; Excel file not created.")
        return

    with pd.ExcelWriter(xls_out, engine="openpyxl") as writer:
        for sheet_name, df in all_dfs.items():
            df.to_excel(
                writer,
                sheet_name=sheet_name[:31],
                index=False,
            )

    print(f"Saved Excel: {xls_out}")


for json_file in sorted(JSON_DIR.glob("*.json")):
    process_json(json_file)

print("Done.")

FULL HTML

In [8]:
JSON_DIR = Path(r"C:\Users\kaustubh.keny\Downloads\OFFICE_FILES\batch_2")
BASE_HTML = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Extracted Tables</title>
<style>
body { font-family: Arial, sans-serif; margin: 30px; }
.source { font-size: 18px; font-weight: bold; margin: 40px 0 20px; }
.figure-title { font-size: 16px; font-weight: bold; margin: 20px 0 10px; }
table { border-collapse: collapse; margin-bottom: 40px; }
td, th { border: 1px solid #999; padding: 6px; }
</style>
</head>
<body>
<h1>Extracted Tables</h1>
"""

all_dfs = {}
html = BASE_HTML

def clean_table(table_html):
    soup = BeautifulSoup(table_html, "html.parser")
    table = soup.find("table")

    if not table:
        return table_html

    total_cols = max(
        sum(int(c.get("colspan", 1)) for c in r.find_all(["td", "th"], recursive=False))
        for r in table.find_all("tr")
    )

    noise_rows = []

    for row in list(table.find_all("tr")):
        cells = row.find_all(["td", "th"], recursive=False)

        if not cells or all(not c.get_text(strip=True) for c in cells):
            row.decompose()
            continue

        if len(cells) == 1 and int(cells[0].get("colspan", 1)) >= total_cols:
            noise_rows.append(cells[0].get_text(" ", strip=True))
            row.decompose()
            continue

        for cell in cells:
            parts = [x.strip() for x in cell.get_text().splitlines() if x.strip()]
            cell.clear()

            for i, part in enumerate(parts):
                if i:
                    cell.append(soup.new_tag("br"))
                cell.append(part)

    result = str(table)

    if noise_rows:
        result += "<br><br><hr><br>" + "".join(
            f"<div>{text}</div>" for text in noise_rows
        )

    return result

def process_json(json_file):
    global html

    print(f"Processing: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for page_data in data["pages"]:
        pdf_name = data["batch"]
        page_num = page_data["page"]

        current_title = None
        page_has_table = False
        table_num = 0

        for block in page_data["blocks"]:
            label = str(block.get("label", "")).lower().strip()
            content = block.get("content", "")

            if label == "figure_title":
                current_title = content
                continue

            if label != "table":
                continue

            table_num += 1

            if not page_has_table:
                html += (
                    f'<div class="source">'
                    f"{pdf_name} | Page {page_num}"
                    f"</div>"
                )
                page_has_table = True

            if current_title:
                html += (
                    f'<div class="figure-title">'
                    f"{current_title}"
                    f"</div>"
                )
                current_title = None

            table = clean_table(content)

            html += f"<div>{table}</div>"

            try:
                tables = pd.read_html(StringIO(table))

                if not tables:
                    continue

                df = tables[0]

            except (ValueError, ImportError) as exc:
                print(
                    f"  Failed: {pdf_name} | "
                    f"Page {page_num} | "
                    f"Table {table_num}: {exc}"
                )
                continue

            df = (
                df.fillna("")
                .map(lambda x: " ".join(str(x).split()))
            )

            normalized_df, metadata = normalize_financial_dataframe(df)

            sheet_name = (
                f"{json_file.stem}_"
                f"p{page_num}_t{table_num}"
            )

            all_dfs[sheet_name] = normalized_df
            
for json_file in sorted(JSON_DIR.glob("*.json")):
    process_json(json_file)

html += "</body></html>"

html_out = JSON_DIR / "all_tables.html"
html_out.write_text(html, encoding="utf-8")

print(f"Saved HTML: {html_out}")


xls_out = JSON_DIR / "all_tables.xlsx"

with pd.ExcelWriter(xls_out, engine="openpyxl") as writer:

    for sheet_name, df in all_dfs.items():

        df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False,
        )

print(f"Saved Excel: {xls_out}")
print("Done.")

Processing: batch_pdl_1.json
Processing: batch_pdl_10.json
Processing: batch_pdl_100.json
Processing: batch_pdl_11.json
Processing: batch_pdl_12.json
Processing: batch_pdl_13.json
Processing: batch_pdl_14.json
Processing: batch_pdl_15.json
Processing: batch_pdl_16.json
Processing: batch_pdl_17.json
Processing: batch_pdl_18.json
Processing: batch_pdl_19.json
Processing: batch_pdl_2.json
Processing: batch_pdl_20.json
Processing: batch_pdl_21.json
Processing: batch_pdl_22.json
Processing: batch_pdl_23.json
Processing: batch_pdl_24.json
Processing: batch_pdl_25.json
Processing: batch_pdl_26.json
Processing: batch_pdl_27.json
Processing: batch_pdl_28.json
Processing: batch_pdl_29.json
Processing: batch_pdl_3.json
Processing: batch_pdl_30.json
Processing: batch_pdl_31.json
Processing: batch_pdl_32.json
Processing: batch_pdl_33.json
Processing: batch_pdl_34.json
Processing: batch_pdl_35.json
  Failed: batch_pdl_35.pdf | Page 2 | Table 1: No tables found matching pattern '.+'
Processing: batch

In [ ]:
from pathlib import Path
import shutil

INPUT_DIR = Path(r"D:\Q1_2026_PDFS")
OUTPUT_DIR = Path(r"D:\Q1_2026_BATCHES")

MAX_BATCH_SIZE = 100 * 1024 * 1024  # 100 MB

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

batch_number = 1
current_batch_size = 0
current_batch_dir = OUTPUT_DIR / f"batch_{batch_number}"
current_batch_dir.mkdir(exist_ok=True)

# No sorting — use filesystem order
files = [f for f in INPUT_DIR.iterdir() if f.is_file()]

for file_path in files:

    file_size = file_path.stat().st_size

    # Start a new batch if adding this file exceeds 100 MB
    if (
        current_batch_size > 0
        and current_batch_size + file_size > MAX_BATCH_SIZE
    ):
        batch_number += 1
        current_batch_size = 0

        current_batch_dir = OUTPUT_DIR / f"batch_{batch_number}"
        current_batch_dir.mkdir(exist_ok=True)

    destination = current_batch_dir / file_path.name

    shutil.copy2(file_path, destination)

    current_batch_size += file_size

    print(
        f"{file_path.name} "
        f"-> batch_{batch_number} "
        f"({current_batch_size / 1024 / 1024:.2f} MB)"
    )

print(f"\nDone. Created {batch_number} batches.")

In [ ]:
from pathlib import Path
import json
import pymupdf

SOURCE_PDF_DIR = Path(r"C:\Q1_2026_RESULTS")
JSON_DIR = Path(r"C:\Users\kaustubh.keny\Downloads\OFFICE_FILES\batch_2\paddle_batch_log.json")
OUTPUT_DIR = Path(r"C:\Users\kaustubh.keny\Downloads\OFFICE_FILES\batch_2")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processing {json_file.name}")

with open(JSON_DIR, "r", encoding="utf-8") as f:
    batches = json.load(f)

# JSON contains a LIST
for batch in batches:

    batch_name = batch["batch_name"]
    output_pdf = OUTPUT_DIR / batch_name
    new_pdf = pymupdf.open()

    for page_info in batch["pages"]:

        source_pdf_name = page_info["source_pdf"]
        original_page = page_info["original_page"]

        crop_rect = page_info["crop_rect"]

        source_pdf_path = SOURCE_PDF_DIR / source_pdf_name

        if not source_pdf_path.exists():
            print(f"Missing PDF: {source_pdf_name}")
            continue

        src_doc = pymupdf.open(source_pdf_path)

        page_index = original_page - 1

        if page_index >= len(src_doc):
            print(
                f"Invalid page {original_page} "
                f"in {source_pdf_name}"
            )
            src_doc.close()
            continue

        src_page = src_doc[page_index]

        # crop rectangle
        rect = pymupdf.Rect(crop_rect)

        # create new page sized to crop
        new_page = new_pdf.new_page(
            width=rect.width,
            height=rect.height,
        )

        # render cropped area
        new_page.show_pdf_page(
            new_page.rect,
            src_doc,
            page_index,
            clip=rect,
        )

        src_doc.close()

    new_pdf.save(output_pdf)
    new_pdf.close()

    print(f"Saved: {output_pdf}")

print("Done.")